# 03 · Pandas: จัดการตารางข้อมูลสถานีวัดน้ำฝน
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apichayaice7835-png/Teach-Python-Geography/blob/main/03_Pandas_Rainfall_Stations.ipynb)

**เวลา:** 3 ชั่วโมง | **ระดับ:** 🟡 ปานกลาง (มีส่วน 🔴 ยาก)

### ข้อมูลที่ใช้ (โฟลเดอร์ `data/`)
| ไฟล์ | เนื้อหา | คอลัมน์ |
|---|---|---|
| `rain_stations_north.csv` | ตำแหน่งสถานี 16 แห่ง ภาคเหนือ | station_id, name_th, name_en, province, lat, lon, elev_m |
| `rain_monthly_2020_2024.csv` | ฝน/อุณหภูมิรายเดือน 5 ปี | station_id, date, rain_mm, tmean_c |

> ⚠️ **ข้อมูลเป็นข้อมูลจำลองเพื่อการสอน** (พิกัดใกล้เคียงอำเภอจริง แต่ค่าฝนสร้างจากแบบจำลอง) มี *ค่าว่าง (missing)* แทรกไว้โดยตั้งใจเพื่อฝึกทำความสะอาดข้อมูล
> เมื่อได้ข้อมูลจริงจากกรมอุตุนิยมวิทยาหรือกรมชลประทาน ให้จัดคอลัมน์ให้ตรงกัน แล้วใช้โค้ดเดิมได้ทันที

**Workflow:** `อ่านไฟล์ → สำรวจ → ทำความสะอาด → รวมตาราง (merge) → สรุป (groupby) → ส่งออก`

In [8]:
# ============================================================
# เซลล์โหลดข้อมูล: หาไฟล์ในเครื่องก่อน → data/ บน GitHub → ชั้นนอกสุดของ repo
# ============================================================
import os
import urllib.request
import pandas as pd

GITHUB_USER = "apichayaice7835-png"
REPO        = "Teach-Python-Geography"
RAW         = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/"

def data_path(filename):
    """คืน path หรือ URL ของไฟล์ที่หาเจอก่อน"""
    for folder in ["data", f"{REPO}/data", "/content/data", "."]:     # 1) ในเครื่อง
        p = os.path.join(folder, filename)
        if os.path.exists(p):
            return p
    for url in [RAW + "data/" + filename, RAW + filename]:              # 2) บน GitHub
        try:
            urllib.request.urlopen(urllib.request.Request(url, method="HEAD"), timeout=10)
            return url
        except Exception:
            pass
    raise FileNotFoundError(f"หาไฟล์ {filename} ไม่เจอ ตรวจว่าอัปโหลดขึ้น GitHub แล้ว")

print("ไฟล์สถานี     :", data_path("rain_stations_north.csv"))
print("ไฟล์ฝนรายเดือน:", data_path("rain_monthly_2020_2024.csv"))

ไฟล์สถานี     : https://raw.githubusercontent.com/apichayaice7835-png/Teach-Python-Geography/main/rain_stations_north.csv
ไฟล์ฝนรายเดือน: https://raw.githubusercontent.com/apichayaice7835-png/Teach-Python-Geography/main/rain_monthly_2020_2024.csv


## 🟢 1. อ่านและสำรวจข้อมูล

In [9]:
stations = pd.read_csv(data_path("rain_stations_north.csv"))
rain     = pd.read_csv(data_path("rain_monthly_2020_2024.csv"))

print("stations:", stations.shape, "| rain:", rain.shape)   # (จำนวนแถว, จำนวนคอลัมน์)
stations.head()

stations: (16, 7) | rain: (960, 4)


,station_id,name_th,name_en,province,lat,lon,elev_m
0,S01,เชียงใหม่,Chiang Mai,Chiang Mai,18.79,98.98,314
1,S02,แม่แจ่ม,Mae Chaem,Chiang Mai,18.50,98.36,480
2,S03,ดอยอ่างขาง,Doi Ang Khang,Chiang Mai,19.93,99.05,1400
3,S04,เชียงราย,Chiang Rai,Chiang Rai,19.91,99.83,390
4,S05,เชียงแสน,Chiang Saen,Chiang Rai,20.27,100.08,360


In [10]:
rain.info()        # ชนิดข้อมูลแต่ละคอลัมน์ และจำนวนค่าที่ไม่ว่าง

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   station_id  960 non-null    object 
 1   date        960 non-null    object 
 2   rain_mm     948 non-null    float64
 3   tmean_c     960 non-null    float64
dtypes: float64(2), object(2)
memory usage: 30.1+ KB


In [11]:
rain.describe()    # สถิติพื้นฐาน: count, mean, std, min, quartiles, max

,rain_mm,tmean_c
count,948.000000,960.000000
mean,107.706962,25.285625
std,139.544710,3.155141
min,0.000000,13.500000
25%,13.675000,23.275000
50%,51.450000,26.200000
75%,160.150000,27.500000
max,941.800000,31.100000


## 🟢 2. เลือกคอลัมน์และกรองแถว

In [12]:
# เลือกคอลัมน์
print(stations[["name_th", "elev_m"]].head(3))

# กรองแถวตามเงื่อนไข (เหมือน Select by Attribute)
high = stations[stations["elev_m"] > 350]
print("\nสถานีที่สูงกว่า 350 ม.:\n", high[["name_th", "province", "elev_m"]])

nan_prov = stations[stations["province"] == "Nan"]
print("\nสถานีในจังหวัดน่าน:", nan_prov["name_th"].tolist())

      name_th  elev_m
0   เชียงใหม่     314
1     แม่แจ่ม     480
2  ดอยอ่างขาง    1400

สถานีที่สูงกว่า 350 ม.:
        name_th    province  elev_m
1      แม่แจ่ม  Chiang Mai     480
2   ดอยอ่างขาง  Chiang Mai    1400
3     เชียงราย  Chiang Rai     390
4     เชียงแสน  Chiang Rai     360
10       พะเยา      Phayao     395

สถานีในจังหวัดน่าน: ['น่าน', 'ท่าวังผา', 'ปัว']


## 🟡 3. จัดการวันที่ (datetime)

In [13]:
rain["date"]  = pd.to_datetime(rain["date"])     # ข้อความ → วันที่
rain["year"]  = rain["date"].dt.year
rain["month"] = rain["date"].dt.month
rain.head()

,station_id,date,rain_mm,tmean_c,year,month
0,S01,2020-01-01,0.0,22.0,2020,1
1,S01,2020-02-01,0.0,23.7,2020,2
2,S01,2020-03-01,13.9,28.0,2020,3
3,S01,2020-04-01,46.4,29.4,2020,4
4,S01,2020-05-01,61.5,27.2,2020,5


## 🟡 4. ทำความสะอาดข้อมูล: ค่าว่าง (Missing values)

In [14]:
print("จำนวนค่าว่างแต่ละคอลัมน์:\n", rain.isna().sum())
print("\nตัวอย่างแถวที่ว่าง:")
rain[rain["rain_mm"].isna()].head()

จำนวนค่าว่างแต่ละคอลัมน์:
 station_id     0
date           0
rain_mm       12
tmean_c        0
year           0
month          0
dtype: int64

ตัวอย่างแถวที่ว่าง:


,station_id,date,rain_mm,tmean_c,year,month
329,S06,2022-06-01,NaN,27.4,2022,6
344,S06,2023-09-01,NaN,27.0,2023,9
356,S06,2024-09-01,NaN,26.8,2024,9
410,S07,2024-03-01,NaN,28.3,2024,3
511,S09,2022-08-01,NaN,27.2,2022,8


In [15]:
# วิธีเติมค่าที่เหมาะกับข้อมูลภูมิอากาศ: ใช้ "ค่าเฉลี่ยของสถานีเดียวกัน เดือนเดียวกัน" (climatological mean)
# transform() คืนค่าที่มีขนาดเท่ากับตารางเดิม จึงเติมกลับได้ทันที
clim = rain.groupby(["station_id", "month"])["rain_mm"].transform("mean")
rain["filled"]  = rain["rain_mm"].isna()          # จดไว้ว่าแถวไหนถูกเติม (โปร่งใสทางวิชาการ)
rain["rain_mm"] = rain["rain_mm"].fillna(clim.round(1))
print("ค่าว่างที่เหลือ:", rain["rain_mm"].isna().sum(), "| จำนวนที่เติม:", rain["filled"].sum())

ค่าว่างที่เหลือ: 0 | จำนวนที่เติม: 12


## 🟡 5. รวมตาราง (merge) = Join ใน GIS
เชื่อมตารางฝนกับตารางสถานีด้วยคอลัมน์ร่วม `station_id`

In [16]:
df = rain.merge(stations, on="station_id", how="left")
df.head()

,station_id,date,rain_mm,tmean_c,year,month,filled,name_th,name_en,province,lat,lon,elev_m
0,S01,2020-01-01,0.0,22.0,2020,1,False,เชียงใหม่,Chiang Mai,Chiang Mai,18.79,98.98,314
1,S01,2020-02-01,0.0,23.7,2020,2,False,เชียงใหม่,Chiang Mai,Chiang Mai,18.79,98.98,314
2,S01,2020-03-01,13.9,28.0,2020,3,False,เชียงใหม่,Chiang Mai,Chiang Mai,18.79,98.98,314
3,S01,2020-04-01,46.4,29.4,2020,4,False,เชียงใหม่,Chiang Mai,Chiang Mai,18.79,98.98,314
4,S01,2020-05-01,61.5,27.2,2020,5,False,เชียงใหม่,Chiang Mai,Chiang Mai,18.79,98.98,314


## 🟡 6. สรุปข้อมูลด้วย `groupby`

In [17]:
# 6.1 ฝนรวมรายปีของแต่ละสถานี
annual = df.groupby(["station_id", "name_th", "year"])["rain_mm"].sum().reset_index()
annual.head()

,station_id,name_th,year,rain_mm
0,S01,เชียงใหม่,2020,791.7
1,S01,เชียงใหม่,2021,1265.4
2,S01,เชียงใหม่,2022,1130.4
3,S01,เชียงใหม่,2023,809.0
4,S01,เชียงใหม่,2024,1236.8


In [18]:
# 6.2 ค่าเฉลี่ยฝนรายปี 5 ปี เรียงจากมากไปน้อย
mean_annual = (annual.groupby(["station_id", "name_th"])["rain_mm"]
                     .mean().round(0)
                     .sort_values(ascending=False)
                     .reset_index(name="mean_annual_mm"))
mean_annual

,station_id,name_th,mean_annual_mm
0,S03,ดอยอ่างขาง,1673.0
1,S14,ท่าวังผา,1563.0
2,S13,น่าน,1516.0
3,S04,เชียงราย,1516.0
4,S05,เชียงแสน,1496.0
5,S11,พะเยา,1384.0
6,S10,ลำพูน,1369.0
7,S08,ลำปาง,1349.0
8,S02,แม่แจ่ม,1240.0
9,S15,ปัว,1232.0


In [19]:
# 6.3 ฝนรายปีเฉลี่ยของทุกสถานี: ปีไหนฝนมาก/น้อย
annual.groupby("year")["rain_mm"].mean().round(0)

,rain_mm
year,
2020,1052.0
2021,1283.0
2022,1688.0
2023,1072.0
2024,1388.0


## 🟡 7. Pivot table: ค่าเฉลี่ยรายเดือน (Climatology) แถว = สถานี, คอลัมน์ = เดือน

In [20]:
climatology = df.pivot_table(index="name_th", columns="month", values="rain_mm", aggfunc="mean").round(0)
climatology

month,1,2,3,4,5,6,7,8,9,10,11,12
name_th,,,,,,,,,,,,
ดอยอ่างขาง,5.0,16.0,25.0,33.0,189.0,219.0,361.0,261.0,339.0,179.0,27.0,19.0
ท่าวังผา,8.0,4.0,24.0,125.0,171.0,183.0,328.0,206.0,332.0,147.0,24.0,10.0
น่าน,3.0,8.0,20.0,66.0,199.0,195.0,333.0,350.0,145.0,125.0,64.0,9.0
ปัว,10.0,6.0,28.0,96.0,139.0,163.0,178.0,187.0,254.0,82.0,82.0,7.0
พะเยา,14.0,10.0,22.0,69.0,191.0,214.0,284.0,96.0,294.0,142.0,40.0,7.0
ลำปาง,3.0,9.0,27.0,77.0,216.0,142.0,234.0,251.0,298.0,58.0,25.0,9.0
ลำพูน,5.0,10.0,29.0,60.0,188.0,175.0,275.0,318.0,166.0,107.0,29.0,8.0
อุตรดิตถ์,6.0,8.0,23.0,69.0,128.0,151.0,233.0,238.0,172.0,80.0,49.0,10.0
เชียงราย,4.0,5.0,21.0,75.0,253.0,179.0,354.0,226.0,245.0,92.0,55.0,6.0


## 🔴 8. จำแนกฤดูกาล และคำนวณ Anomaly
ฤดูกาลแบบประมาณตามเดือน: ฤดูหนาว (พ.ย.–ก.พ.), ฤดูร้อน (มี.ค.–เม.ย.), ฤดูฝน (พ.ค.–ต.ค.)

In [21]:
season_map = {11:"หนาว", 12:"หนาว", 1:"หนาว", 2:"หนาว",
              3:"ร้อน", 4:"ร้อน",
              5:"ฝน", 6:"ฝน", 7:"ฝน", 8:"ฝน", 9:"ฝน", 10:"ฝน"}
df["season"] = df["month"].map(season_map)

season_share = df.pivot_table(index="name_th", columns="season", values="rain_mm", aggfunc="sum")
season_share = (season_share.div(season_share.sum(axis=1), axis=0) * 100).round(1)   # เป็นร้อยละ
season_share.sort_values("ฝน", ascending=False)

season,ฝน,ร้อน,หนาว
name_th,,,
ดอยอ่างขาง,92.5,3.5,4.0
แม่ฮ่องสอน,90.1,7.0,3.0
ลำพูน,89.7,6.5,3.8
เชียงแสน,89.3,6.4,4.2
เชียงราย,89.0,6.3,4.6
ลำปาง,88.9,7.7,3.4
แพร่,88.8,5.7,5.5
น่าน,88.8,5.6,5.5
พะเยา,88.3,6.6,5.1


In [22]:
# Anomaly รายปี (%) = (ฝนปีนั้น − ค่าเฉลี่ย 5 ปี) / ค่าเฉลี่ย × 100
annual["mean5"] = annual.groupby("station_id")["rain_mm"].transform("mean")
annual["anomaly_pct"] = ((annual["rain_mm"] - annual["mean5"]) / annual["mean5"] * 100).round(1)
anom_table = annual.pivot(index="name_th", columns="year", values="anomaly_pct")
anom_table

year,2020,2021,2022,2023,2024
name_th,,,,,
ดอยอ่างขาง,-13.7,4.9,43.3,-39.9,5.3
ท่าวังผา,-19.2,-33.4,17.5,17.6,17.6
น่าน,-40.6,-28.2,77.2,-24.5,16.2
ปัว,3.8,-23.9,10.5,-3.1,12.7
พะเยา,-42.0,5.6,38.0,-9.3,7.7
ลำปาง,30.0,-24.2,-5.4,-44.3,44.0
ลำพูน,-30.9,2.9,16.2,-22.5,34.3
อุตรดิตถ์,-20.9,26.3,63.7,-30.3,-38.8
เชียงราย,-2.0,6.8,50.9,-34.7,-21.0


In [25]:
#  ความสัมพันธ์ระหว่างความสูงกับฝนรายปี (Pearson correlation)
merged = mean_annual.merge(stations[["station_id", "elev_m", "lon"]], on="station_id")
print("r(ความสูง, ฝน)     = %.2f" % merged["mean_annual_mm"].corr(merged["elev_m"]))
print("r(ลองจิจูด, ฝน)    = %.2f" % merged["mean_annual_mm"].corr(merged["lon"]))
print(" Correlation ≠ Causation และตัวอย่างมีเพียง 16 สถานี")

r(ความสูง, ฝน)     = 0.54
r(ลองจิจูด, ฝน)    = 0.45
 Correlation ≠ Causation และตัวอย่างมีเพียง 16 สถานี


## 9. ส่งออกผลลัพธ์

In [24]:
os.makedirs("output", exist_ok=True)
mean_annual.merge(stations, on=["station_id", "name_th"]).to_csv(
    "output/station_mean_annual_rain.csv", index=False, encoding="utf-8-sig")   # utf-8-sig เปิดใน Excel แล้วภาษาไทยไม่เพี้ยน
print("บันทึกแล้ว → output/station_mean_annual_rain.csv")

บันทึกแล้ว → output/station_mean_annual_rain.csv


## 📎 เมื่อใช้ไฟล์จริงของตนเอง (เคล็ดลับ)
```python
pd.read_csv("file.csv", encoding="cp874")        # ไฟล์ภาษาไทยจาก Windows/Excel รุ่นเก่า
pd.read_excel("file.xlsx", sheet_name="2567")    # ไฟล์ Excel
df = df.rename(columns={"รหัสสถานี":"station_id", "ละติจูด":"lat", "ลองจิจูด":"lon"})
df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")   # ค่าอย่าง "T" (trace), "-" → NaN
```
> ข้อมูลฝนรายวันจากหน่วยงานมักใช้ `T` แทน *ฝนเล็กน้อยวัดไม่ได้* (< 0.1 มม.) ควรตัดสินใจให้ชัดว่าจะแทนด้วย 0 หรือ NaN

---
## ✏️ แบบฝึกหัด
**🟢** 1) หาสถานีที่อยู่ต่ำที่สุด  2) กรองข้อมูลเฉพาะปี 2022 ของสถานีน่าน (S13)

**🟡** 3) หาเดือนที่ฝนมากที่สุดของแต่ละสถานี (hint: `idxmax`)
4) คำนวณอุณหภูมิเฉลี่ยรายปีของแต่ละจังหวัด (`groupby("province")`)

**🔴** 5) นับจำนวน "เดือนแล้ง" (ฝน < 50 มม.) ต่อปีของแต่ละสถานี แล้วหาว่าปีไหนแล้งที่สุด
6) หาสถานีที่มีค่าว่างมากที่สุด และอภิปรายว่าการเติมค่าด้วยค่าเฉลี่ยมีข้อจำกัดอะไร